# NQ Intraday Breakout — backtest

The engine lives in the [`nq_breakout`](src/nq_breakout) package; this notebook
only loads data, runs it and reports. Rules were frozen in 2022 (original
MultiCharts strategy); the Python engine is a 2025 port, not yet validated
trade-by-trade against the original.

**The cells below run both the corrected stop-fill configurations and the
actual legacy configuration.** The README reports all three, not one single
answer. The same
rules on the same data give Sharpe 0.95 under the original notebook's
conventions and 1.11 under corrected conventions with close fills — and most of
that spread comes from the entry-fill assumption, which is still unresolved.
See [`docs/bias-audit.md`](docs/bias-audit.md) for the twelve differences one at
a time, and run `tools/bias_attribution.py` to reproduce the whole chain.

Before trusting any number, read the **data diagnostics** cell output: it checks
the feed's timezone, roll splicing and overnight gap exposure.

In [ ]:
import pandas as pd
from dataclasses import replace
from nq_breakout import (LEGACY_CONFIG_KWARGS, LEGACY_FLAGS, Config,
                         legacy_equity_curve, load_nq_1min, load_qqq, resample_5min,
                         run_data_diagnostics, run_backtest, metrics_table,
                         format_metrics, align_benchmark, trade_pnl_tstat)
from nq_breakout.plots import plot_backtest, plot_vs_benchmark

BASE = Config()  # risk 1%, entry_mode='stop' — see backtest.Config for every knob
BASE

In [ ]:
nq_1min = load_nq_1min('Data/nq-10y-1min.csv')
qqq = load_qqq('Data/QQQ_daily.csv')
df_5min = resample_5min(nq_1min)

run_data_diagnostics(nq_1min, BASE)

In [ ]:
bt_1pct = run_backtest(df_5min, replace(BASE, risk_pct=0.01))
bt_2pct = run_backtest(df_5min, replace(BASE, risk_pct=0.02))
legacy_config = replace(BASE, risk_pct=0.01, **LEGACY_CONFIG_KWARGS)
bt_legacy = run_backtest(df_5min, legacy_config, LEGACY_FLAGS)
legacy_equity = legacy_equity_curve(bt_legacy.trades, legacy_config.initial_capital)

for name, bt in [('1% risk, stop entry ', bt_1pct),
                 ('2% risk, stop entry ', bt_2pct),
                 ('1% risk, legacy conventions', bt_legacy)]:
    t = bt.trades
    stats = trade_pnl_tstat(t)
    print(f"{name}: {len(t):5d} trades | net PnL {t['pnl'].sum():14,.0f} | "
          f"avg contracts {t['contracts'].mean():6.2f} | "
          f"PnL/contract {stats['mean']:8.2f} (t-stat {stats['t_stat']:.2f})")

In [ ]:
plot_backtest(bt_1pct, title_prefix='1% risk')

qqq_equity = align_benchmark(qqq, bt_1pct.equity.index, BASE.initial_capital)
plot_vs_benchmark(bt_1pct, qqq_equity)

In [ ]:
split_date = pd.Timestamp('2022-01-31')  # rules frozen in early 2022

equities = {
    'NQ 1% risk (stop entry)': bt_1pct.equity,
    'NQ 2% risk (stop entry)': bt_2pct.equity,
    'NQ 1% risk (legacy conventions)': legacy_equity,
    'QQQ Buy & Hold': qqq_equity,
}

print("Overall (each configuration's published equity convention):")
display(format_metrics(metrics_table(equities)))

print(f'Pre-split (before {split_date.date()}):')
display(format_metrics(metrics_table({k: v[v.index < split_date] for k, v in equities.items()})))

print(f'Post-split (from {split_date.date()}):')
display(format_metrics(metrics_table({k: v[v.index >= split_date] for k, v in equities.items()})))

In [ ]:
# How do the corrected stop-fill and full legacy conventions differ by direction?
comparison = pd.DataFrame({
    'stop entry': bt_1pct.trades.groupby('direction')['pnl_per_contract'].agg(['mean', 'count']).unstack(),
    'legacy conventions': bt_legacy.trades.groupby('direction')['pnl_per_contract'].agg(['mean', 'count']).unstack(),
}).round(2)
print('PnL per contract by direction and simulation convention:')
display(comparison)